In [10]:
%cd /content/drive/MyDrive/SHIELD_DOC
!git pull origin feature/document-security

/content/drive/MyDrive/SHIELD_DOC
From https://github.com/jaeyong3126/SHIELD_DOC
 * branch            feature/document-security -> FETCH_HEAD
Already up to date.


In [11]:
from google.colab import drive
drive.mount('/content/drive')

# 1. 드라이브 최상단으로 이동
%cd /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive


In [9]:
!pip install pypdf python-docx

In [36]:
import sys
import os
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import re

# 파이썬이 SHIELD_DOC 폴더를 인식할 수 있도록 경로 추가
sys.path.append('/content/drive/MyDrive/SHIELD_DOC')

# tools 폴더 안의 parser 파일에서 함수 불러오기
from tools.parser import parse_document

# 생성된 Confidential데이터들로 텍스트, 라벨 칼럼을 생성하여 데이터 프레임 변환 및 confidential_dataset.csv 파일 생성
data_list = []

# 기밀 문서 폴더 경로 설정 (실제 경로에 맞게 수정 필요 시 수정)
confidential_dir = '/content/drive/MyDrive/Confidential/'

print("tools 폴더의 파서를 성공적으로 불러왔습니다. 데이터 파싱을 시작합니다...")

# 폴더 내 파일 순차적 읽기 및 파싱
if os.path.exists(confidential_dir):
    for filename in os.listdir(confidential_dir):
        file_path = os.path.join(confidential_dir, filename)

        try:
            # 팀원의 파서 함수 실행
            parsed_data = parse_document(file_path)

            # 파서가 정상적으로 텍스트를 뽑아왔다면 리스트에 추가 (라벨 1)
            if parsed_data["text"].strip():
                data_list.append({
                    "filename": parsed_data["filename"],
                    "text": parsed_data["text"],
                    "label": 1  # 기밀 문서는 1
                })
        except Exception as e:
            print(f"파싱 실패 ({filename}): {e}")
else:
    print("지정하신 경로에 confidential 폴더가 없습니다. 경로를 확인해주세요.")

# DataFrame 변환 및 CSV 저장
if data_list:
    df_confidential = pd.DataFrame(data_list)

    csv_filename = '/content/drive/MyDrive/confidential_dataset.csv'
    df_confidential.to_csv(csv_filename, index=False, encoding="utf-8-sig")

    print("-" * 50)
    print(f" 데이터셋 구축 완료 총 {len(df_confidential)}건의 문서가 처리되었습니다.")
    display(df_confidential.head())  # 상위 5개 미리보기
    print("-" * 50)

    # 생성된 Normal 데이터들로 텍스트, 라벨 칼럼을 생성하여 데이터 프레임 변환 및 normal_dataset.csv 파일 생성

data_list = []

normal_data_list = []

# 정상 문서 폴더 경로 설정
# (드라이브 내에 normal 폴더명에 맞춰 경로를 수정해 주세요)
normal_dir = '/content/drive/MyDrive/Normal/'

print("파서를 사용하여 정상 문서(Normal) 파싱을 시작합니다...")

if os.path.exists(normal_dir):
    for filename in os.listdir(normal_dir):
        file_path = os.path.join(normal_dir, filename)

        try:
            # 팀원의 파서 함수로 문서 텍스트 추출
            parsed_data = parse_document(file_path)

            # 텍스트가 정상적으로 존재하면 리스트에 추가 (라벨 0: 정상/허용)
            if parsed_data["text"].strip():
                normal_data_list.append({
                    "filename": parsed_data["filename"],
                    "text": parsed_data["text"],
                    "label": 0  # 정상 문서는 라벨 0
                })
        except Exception as e:
            print(f"파싱 실패 ({filename}): {e}")
else:
    print(f"지정하신 경로에 normal 폴더가 없습니다: {normal_dir}")

# DataFrame 변환 및 CSV 저장
if normal_data_list:
    df_normal = pd.DataFrame(normal_data_list)

    csv_filename = '/content/drive/MyDrive/normal_dataset.csv'
    df_normal.to_csv(csv_filename, index=False, encoding="utf-8-sig")

    print("-" * 50)
    print(f"정상 문서 데이터셋 구축 완료! 총 {len(df_normal)}건의 문서가 처리되었습니다.")
    display(df_normal.head())  # 상위 5개 미리보기 확인
    print(f"저장된 파일 경로: {csv_filename}")
    print("-" * 50)
else:
    print("처리된 데이터가 없습니다. 폴더 안의 파일 형식을 확인해주세요.")

# concat으로 두 개의 데이터 하나로 합치기

# 각각 저장해 둔 CSV 파일 불러오기
conf_path = '/content/drive/MyDrive/confidential_dataset.csv'
norm_path = '/content/drive/MyDrive/normal_dataset.csv'

df_conf = pd.read_csv(conf_path)
df_norm = pd.read_csv(norm_path)

print(f"기밀 문서 데이터: {len(df_conf)}건")
print(f"정상 문서 데이터: {len(df_norm)}건")

# concat으로 두 데이터 합치기
df_datasets = pd.concat([df_conf, df_norm], ignore_index=True)
print(f"Concat 통합 완료! 전체 데이터 총 {len(df_datasets)}건")
df_datasets.head()

# EDA 진행 (결측치 확인 및 라벨 비율 확인)

print('결측치 확인')
print(df_datasets.isnull().sum())
percent = df_datasets['label'].value_counts(normalize=True) * 100
print('\n라벨(Label) 비율 확인')
print(percent)
# 결측치는 없으며, 라벨 비율이 대략 53%:47% 이므로 가중치를 주지 않아도 된다고 판단.

try:
    from tools.pii_detector import detect_pii
except ImportError:
    print("모듈을 찾을 수 없습니다. 경로를 확인해주세요.")

# 텍스트 전처리

def clean_text(text):
    if pd.isna(text) or not text:
        return ''

    # 한글, 영문, 숫자, 마스킹 태그([], _)를 제외한 모든 특수문자 제거
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\[\]_]', ' ', str(text))

    # 연속된 공백이나 줄바꿈을 하나의 공백으로 압축
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# 3. 데이터에 마스킹 및 전처리 일괄 적용

processed_data = []

# 데이터프레임의 각 행을 순회하며 적용
for index, row in df_datasets.iterrows():
    original_text = row['text']

    # 모듈로 개인정보 마스킹 처리
    pii_result = detect_pii(original_text)
    masked_text = pii_result.get("masked_text", original_text)

    # 전처리 함수로 노이즈 정제
    final_cleaned_text = clean_text(masked_text)

    processed_data.append(final_cleaned_text)


# 4. 최종 데이터셋 저장
# 전처리 완료된 텍스트를 새로운 컬럼으로 추가
df_datasets['cleaned_text'] = processed_data

# 정제 후 결측치 제거
df_final = df_datasets[df_datasets['cleaned_text'] != '']

# 최종 모델 학습용 CSV로 저장
final_csv_path = '/content/drive/MyDrive/final_preprocessed_dataset.csv'
df_final.to_csv(final_csv_path, index=False, encoding='utf-8-sig')

print(f'전처리 완료! 총 {len(df_final)}건의 텍스트가 전처리되었습니다.')
print(f'저장 경로: {final_csv_path}\n')
df_final_1 = df_final[['filename', 'label', 'cleaned_text']]
final_csv_path = '/content/drive/MyDrive/final_preprocessed_dataset_1.csv'
df_final_1.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
# 최종 결과 상위 5개 미리보기 (라벨과 정제된 텍스트 확인)
display(df_final.head())
display(df_final_1.head())


tools 폴더의 파서를 성공적으로 불러왔습니다. 데이터 파싱을 시작합니다...
--------------------------------------------------
 데이터셋 구축 완료 총 590건의 문서가 처리되었습니다.


,filename,text,label
0,design_arw_028.txt,﻿[한빛반도체 내부 문서]\n제목: HB-NAND X3 read reference ...,1
1,design_cvy_043.txt,﻿[한빛반도체 내부 문서]\n제목: HB-SER S6 CDR bandwidth 기술...,1
2,design_cyd_101.txt,﻿[한빛반도체 내부 문서]\n제목: HB-NAND A4 peak temperatur...,1
3,design_bsh_025.txt,﻿[한빛반도체 내부 문서]\n제목: HB-DRAM D9 redundancy 연구 주...,1
4,design_cje_023.txt,﻿[한빛반도체 내부 문서]\n제목: HB-DRAM D9 refresh interva...,1


--------------------------------------------------
파서를 사용하여 정상 문서(Normal) 파싱을 시작합니다...
--------------------------------------------------
정상 문서 데이터셋 구축 완료! 총 517건의 문서가 처리되었습니다.


,filename,text,label
0,design_pmd_137_edge.txt,﻿[한빛반도체 연구·설계 문서]\n제목: Mask CD 변동성 검증 공정 사양서\n...,0
1,design_fvz_138_edge.txt,﻿[한빛반도체 연구·설계 문서]\n제목: DRAM retention fail 분석 ...,0
2,design_qrq_135_edge.txt,﻿[한빛반도체 연구·설계 문서]\n제목: Wafer defect map 분석 연구 ...,0
3,edge_mju_168.txt,[사내 교육자료] 수율 관리의 기본 개념\n작성부서: 품질보증팀 (내선 3635)\...,0
4,edge_hgb_169.txt,[경영진 보고] 12월 임원 참석 행사 진행안\n작성부서: 커뮤니케이션팀 (내선 2...,0


저장된 파일 경로: /content/drive/MyDrive/normal_dataset.csv
--------------------------------------------------
기밀 문서 데이터: 590건
정상 문서 데이터: 517건
Concat 통합 완료! 전체 데이터 총 1107건
결측치 확인
filename    0
text        0
label       0
dtype: int64

라벨(Label) 비율 확인
label
1    53.2972
0    46.7028
Name: proportion, dtype: float64
전처리 완료! 총 1107건의 텍스트가 전처리되었습니다.
저장 경로: /content/drive/MyDrive/final_preprocessed_dataset.csv



,filename,text,label,cleaned_text
0,design_arw_028.txt,﻿[한빛반도체 내부 문서]\n제목: HB-NAND X3 read reference ...,1,[한빛반도체 내부 문서] 제목 HB NAND X3 read reference 기술 ...
1,design_cvy_043.txt,﻿[한빛반도체 내부 문서]\n제목: HB-SER S6 CDR bandwidth 기술...,1,[한빛반도체 내부 문서] 제목 HB SER S6 CDR bandwidth 기술 회의...
2,design_cyd_101.txt,﻿[한빛반도체 내부 문서]\n제목: HB-NAND A4 peak temperatur...,1,[한빛반도체 내부 문서] 제목 HB NAND A4 peak temperature 실...
3,design_bsh_025.txt,﻿[한빛반도체 내부 문서]\n제목: HB-DRAM D9 redundancy 연구 주...,1,[한빛반도체 내부 문서] 제목 HB DRAM D9 redundancy 연구 주간보고...
4,design_cje_023.txt,﻿[한빛반도체 내부 문서]\n제목: HB-DRAM D9 refresh interva...,1,[한빛반도체 내부 문서] 제목 HB DRAM D9 refresh interval 기...


,filename,label,cleaned_text
0,design_arw_028.txt,1,[한빛반도체 내부 문서] 제목 HB NAND X3 read reference 기술 ...
1,design_cvy_043.txt,1,[한빛반도체 내부 문서] 제목 HB SER S6 CDR bandwidth 기술 회의...
2,design_cyd_101.txt,1,[한빛반도체 내부 문서] 제목 HB NAND A4 peak temperature 실...
3,design_bsh_025.txt,1,[한빛반도체 내부 문서] 제목 HB DRAM D9 redundancy 연구 주간보고...
4,design_cje_023.txt,1,[한빛반도체 내부 문서] 제목 HB DRAM D9 refresh interval 기...
